## Retreiving the Bronze table

In [0]:
df_bronze = spark.table("climate_project.bronze_observations")
print(df_bronze.count())

391479581


## Filtering out the three elements that we are going to focus.

In [0]:
core_elements = ["TMAX", "TMIN", "PRCP"]
df_filtered = df_bronze.filter(df_bronze.ELEMENT.isin(core_elements))

print(df_filtered.count())

210181311


## Quality Filter
### Q_FLAG marks readings NOAA's own QA checks flagged as suspect. Keeping only rows where it's null keeps just the good-quality data.

In [0]:
df_clean = df_filtered.filter(df_filtered.Q_FLAG.isNull())

print(df_clean.count())

209958757


%md
## Fix Types
### Bronze columns come in as all strings. Cast `DATE` to an actual date and `DATA_VALUE` to an integer so we can do date math and numeric operations downstream.

In [0]:
from pyspark.sql.functions import col, to_date

df_typed = df_clean.withColumn("DATE", to_date(col("DATE"), "yyyyMMdd")) \
                    .withColumn("DATA_VALUE", col("DATA_VALUE").cast("int"))

df_typed.printSchema()

root
 |-- ID: string (nullable = true)
 |-- DATE: date (nullable = true)
 |-- ELEMENT: string (nullable = true)
 |-- DATA_VALUE: integer (nullable = true)
 |-- M_FLAG: string (nullable = true)
 |-- Q_FLAG: string (nullable = true)
 |-- S_FLAG: string (nullable = true)
 |-- OBS_TIME: string (nullable = true)



## Raw values are in tenths of a unit (e.g., 252 = 25.2°C). Dividing by 10 converts to real-world °C/mm.

In [0]:
df_scaled = df_typed.withColumn("DATA_VALUE", col("DATA_VALUE") / 10.0)

df_scaled.show(10, truncate=False)

+-----------+----------+-------+----------+------+------+------+--------+
|ID         |DATE      |ELEMENT|DATA_VALUE|M_FLAG|Q_FLAG|S_FLAG|OBS_TIME|
+-----------+----------+-------+----------+------+------+------+--------+
|AE000041196|2023-01-01|TMAX   |25.2      |NULL  |NULL  |S     |NULL    |
|AE000041196|2023-01-01|TMIN   |14.9      |NULL  |NULL  |S     |NULL    |
|AE000041196|2023-01-01|PRCP   |0.0       |D     |NULL  |S     |NULL    |
|AEM00041194|2023-01-01|TMAX   |25.5      |NULL  |NULL  |S     |NULL    |
|AEM00041194|2023-01-01|TMIN   |18.6      |NULL  |NULL  |S     |NULL    |
|AEM00041194|2023-01-01|PRCP   |0.0       |NULL  |NULL  |S     |NULL    |
|AEM00041217|2023-01-01|TMAX   |24.8      |NULL  |NULL  |S     |NULL    |
|AEM00041217|2023-01-01|TMIN   |18.4      |NULL  |NULL  |S     |NULL    |
|AEM00041218|2023-01-01|TMAX   |25.4      |NULL  |NULL  |S     |NULL    |
|AEM00041218|2023-01-01|TMIN   |14.5      |NULL  |NULL  |S     |NULL    |
+-----------+----------+-------+------

## Saving the Delta table.

In [0]:
df_scaled.write.format("delta").mode("overwrite").saveAsTable("climate_project.silver_observations")

## NOAA's station file is fixed-width text, so each field is sliced out by exact character position. These positions come directly from NOAA's official format documentation for `ghcnd-stations.txt`.

In [0]:
stations_raw = spark.read.text("s3://noaa-ghcn-pds/ghcnd-stations.txt")

from pyspark.sql.functions import trim

stations = stations_raw.select(
    trim(stations_raw.value.substr(1, 11)).alias("ID"),
    trim(stations_raw.value.substr(13, 8)).cast("double").alias("LATITUDE"),
    trim(stations_raw.value.substr(22, 9)).cast("double").alias("LONGITUDE"),
    trim(stations_raw.value.substr(32, 6)).cast("double").alias("ELEVATION"),
    trim(stations_raw.value.substr(39, 2)).alias("STATE"),
    trim(stations_raw.value.substr(42, 30)).alias("NAME")
)

stations.write.format("delta").mode("overwrite").saveAsTable("climate_project.silver_stations")

print(stations.count())

132503
